# Técnicas avanzadas de diseño de software


## MODIN - 모든
A drop-in replacement for pandas


### Gonzalo Giarda

<div>
<img src="img/ducksis (3).svg"/>
</div>





# Nacimiento 

A partir de una publicación en un blog en 2018 sobre un *'hack'* de pandas en Ray, desarrollado por el estudiante de doctorado **Devin Petersohn** en colaboración con otros estudiantes de grado de la Universidad de Berkeley, surgió un interés que eventualmente dio origen al proyecto Modin, impulsado por el **RISELab**. 

Puntalmente teniendo problemas con la **escalabilidad de pandas** a un gran volumen de datos (procesamiento de datos genomico).

# El abismo

👩‍🔬 Usuario objetivo: Científicos de datos posiblemente sin conociemientos en computación distribuida.

🛠️ Herramientas comunes:

- Pandas, R, Jupyter Notebook

- Evaluación inmediata y trabajo interactivo

🚧 Desafío:

- Escalar a grandes volúmenes requiere herramientas distintas (Spark, Dask, TensorFlow)

- Estas demandan conocimientos técnicos adicionales (particiones, flujos de datos, ejecución diferida)

<div>
<img src="img/abismo.jpeg"/>
</div>


🎯 Objetivo de Modin:

- Mantener la experiencia de usuario de Pandas

- Escalar sin requerir cambios significativos en el código ni conocimiento avanzado en cómputo distribuido

# pandas vs modin
## Escalabilidad

<table><tr>
<td> <img src="img/pandas_multicore.png"/> </td>
<td> <img src="img/modin_multicore.png"/> </td>
</tr></table>

Modin ofrece la **misma interfaz de Pandas** a través de modin.pandas, pero evita las limitaciones que dificultan escalar el rendimiento.
Mientras que Pandas es de un solo hilo (single-threaded) y solo utiliza un núcleo del procesador, Modin puede usar todos los núcleos disponibles en tu máquina, o incluso todos los núcleos de un clúster completo.

# pandas vs modin
## Uso de memoria e inmutabilidad

La API de Pandas permite muchas operaciones "inplace", pero aunque parezca que se ahorra memoria, en realidad Pandas casi siempre copia los datos internamente.

Modin, en cambio, simula el comportamiento inplace, pero internamente sus estructuras de datos son **inmutables**. Esto le permite:

- Encadenar operaciones de forma más eficiente

- Gestionar mejor la memoria

- Compartir bloques de memoria entre distintos dataframes

Para mantener la compatibilidad con la API de Pandas, Modin usa **punteros mutables** a estructuras inmutables. Así, cuando haces una operación inplace, Modin simplemente actualiza ese puntero al nuevo dataframe.

# pandas vs modin

La API de **Pandas** es conocida por tener múltiples formas de realizar una misma operación, lo que lleva a una implementación compleja y redundante.

Modin, en cambio, impone una regla clara: **una sola implementación por operación**. Esta simplificación permite:

- Reducir la complejidad interna del código

- Optimizar de forma más efectiva

- Mantener compatibilidad total con la API de Pandas

Internamente, Modin usa una **álgebra reducida de operadores**: un conjunto reducido de unas 15 operaciones fundamentales, derivadas de más de 200 en Pandas. Este diseño está basado tanto en fundamentos teóricos como en aplicaciones prácticas.

>El nombre **모든/MODIN/TODIN** es derivado de la palabra coreana para  “todo”, pues su objetivo son todos los operadores de un dataframe.

# pandas vs modin
## Ejecución Fuera de Núcleo (Out-of-Core) con Modin

- Pandas está diseñado para trabajar solo con datos que **caben completamente en memoria RAM**

- Modin puede procesar datos que exceden la memoria mediante la **ejecución fuera de núcleo**. Esta funcionalidad permite que Modin maneje conjuntos de datos más grandes que la RAM disponible utilizando mecanismos como la escritura en disco.

- A través de sus motores de ejecución distribuidos, como Ray o Dask, los cuales pueden manejar operaciones distribuidas y realizar el **spilling** (derramar datos) al disco cuando la memoria es insuficiente.

- Cuando Modin detecta que no hay suficiente memoria, automáticamente intercambia partes de los datos a disco para mantener la ejecución sin fallos. Esto es especialmente útil en entornos donde los conjuntos de datos pueden crecer dinámicamente o no se puede garantizar que todos los datos quepan en la memoria.

# Aqruitectura 

<div>	
<img src="img/modin_architecture (1).png"/>
</div>

Modin está estructurado en capas, similares a la jerarquía de un sistema de gestión de bases de datos (DBMS). Esta separación modular permite:

- Optimizar cada componente idividualmente

- Reemplazar o mejorar partes del sistema sin afectar el resto

La lógica de distribución permanece independiente y es gestionada internamente por el motor de cómputo elegido (Ray, Dask, etc.).

# Subsystem/Container View

Ejemplo para ray

<div>
<img src="img/component_view.png"/>
</div>

1. 🧠 DataFrame Subsystem

    - Traduce llamadas Pandas al álgebra interna

    - Compilación de consultas

2. 📥📤 Ingress/Egress Module

    - Divide y envía datos por particiones

    - Interactúa con nodos de almacenamiento

3. 🗺️ Query Planner

    - Convierte API de Pandas en DAG del álgebra
    
    - Realiza optimizaciones iniciales

4. ⚙️ Query Executor

    - Optimiza y ejecuta el DAG según el formato
    
    - Mapea a una secuencia ejecutable

5. 🧾 Storage Formats Module

    - Traduce operaciones abstractas a ejecución real
    
    - Soporta múltiples formatos (Pandas, personalizados)

6. 🚀 Orchestration Subsystem

    - Lanza nodos y motores (ej. Ray)
    
    - Monitorea estado y da telemetría

# Componentes - Data Transformation 
## Query Compiler


El **Query Compiler** en Modin recibe las consultas desde la capa de la API de Pandas, que se encarga de entregar entradas limpias y normalizadas.

Este componente debe tener conocimiento tanto de los kernels de cómputo como de la representación en memoria de los datos para poder compilar eficientemente la consulta.

<div style="display: flex; gap:10px;">
  <img src="img/querycompiler1.svg" alt="img"/>
</div>

Una vez compilada, la consulta se envía al núcleo del DataFrame de Modin. Importante: el Query Compiler no decide cuándo ni dónde se ejecutará la consulta. Tampoco maneja la distribución de las particiones, eso lo hace el DataFrame interno.

Además, el Query Compiler replica la API de Pandas pero evita duplicaciones innecesarias, simplificando la interfaz interna.

# Componentes - Data Transformation
## Core Modin Dataframe

💤 Soporte para ejecución perezosa (lazy)

- Actualmente: mayoría de operaciones se ejecutan como en Pandas (eager)

- Casos costosos como transpose pueden diferirse

🔧 Estructura extensible

- Operaciones pueden ser optimizadas según el motor de ejecución

🧹 API reducida y unificada

- Una única forma de ejecutar cada operación

- Más simple que la API de QueryCompiler o Pandas

## Core Modin Dataframe API

- El layout (distribución) de los datos

- La reorganización (shuffling) de los datos entre particiones

- La partición de los datos en fragmentos manejables

- La serialización de las tareas que se envían a cada partición para su ejecución

# Componentes - Data Transformation
## Partition Manager

El Partition Manager puede modificar el tamaño y la forma de las particiones **según el tipo de operación**. Por ejemplo, algunas operaciones complejas necesitan acceder a una columna o fila completa. El Partition Manager puede transformar particiones en bloques a particiones por filas o columnas, dando a Modin la flexibilidad para manejar operaciones que serían difíciles con solo particiones por filas o columnas.

Además, el Partition Manager se encarga de **serializar y enviar las consultas compiladas a las particiones correspondientes**. Mantiene metadatos sobre el tamaño (alto y ancho) de cada partición, por lo que cuando una operación afecta solo a una parte del DataFrame, la consulta se envía directamente a la partición correcta. Esto es fundamental para operaciones en pandas que aplican diferentes argumentos a distintas columnas.

Esta abstracción permite separar la movilización de datos y la aplicación de funciones de la capa del DataFrame, manteniendo la API central de Modin pequeña y permitiendo optimizar por separado el movimiento de datos y la gestión de metadatos.

<div>
<img src="img/partition.png"/>
</div>

## Partitions
Las Partitions (particiones) gestionan un subconjunto del DataFrame. El DataFrame está particionado tanto por filas como por columnas, lo que brinda a Modin escalabilidad en ambas direcciones y flexibilidad en el diseño de los datos.

Modin implementa varias optimizaciones a nivel de particiones. Estas particiones son **específicas del framework de ejecución** y del formato en memoria de los datos, lo que permite aprovechar optimizaciones tanto en el procesamiento como en el almacenamiento.

# Componentes - Execution Engine
Esta capa se encarga de realizar el **cómputo sobre las particiones** de los datos.

# Componentes - Storage Format
El formato de almacenamiento describe el **tipo de partición en memoria**.

El formato base en Modin es pandas. Por defecto, el DataFrame de Modin opera sobre particiones que contienen objetos de tipo pandas.DataFrame. 

# Componentes - Data Ingress


<div>
<img src="img/ingress.svg"/>
</div>

El ingreso de datos comienza con una función en la capa de la API de pandas (por ejemplo, read_csv). Luego, la consulta del usuario se envía al Factory Dispatcher, que define una fábrica específica para el motor de ejecución.

Esta fábrica contiene una clase de entrada/salida (IO) encargada de realizar la lectura/escritura paralela desde/hacia un archivo. Esta clase IO incluye métodos de clase con interfaces y nombres similares a las funciones de IO de pandas, como read_csv.

La clase IO también:

- Declara las clases específicas del Modin Dataframe y del Query Compiler para ese motor y formato de almacenamiento.

- Define métodos de IO que combinan:

    - El motor de ejecución para tareas remotas.

    - El analizador del formato del archivo.

    - El gestor de fragmentación del archivo en el nodo principal.

El resultado final de la función de ingreso de datos de esta clase IO es un Modin Dataframe.

# Componentes - Data Egress

<div>
<img src="img/egress.svg"/>
</div>

Las operaciones de egreso de datos (por ejemplo, to_csv) son similares a las de ingreso, hasta el punto de construir las funciones específicas del motor en la clase IO.

Sin embargo, las funciones de egreso en la clase IO están definidas de forma ligeramente diferente a las de ingreso y se crean **específicamente para el motor de ejecución**, ya que las particiones **ya conocen su formato de almacenamiento**.

Usando esta clase IO, los datos se exportan desde las particiones hacia el archivo de destino.

# Algebra de los Dataframes
Un datafeame es una tupla $(A_{m\times n},R_m,C_n,D_n)$ donde $A_{m\times n}$ es un array $m\times n$, $R_m$ un array de $m$ labels oara las filas, $C_n$ un arrray para de $n$ labels para las columnas y $D_n$ es el array de tiupos de cada columna.

<div>
<img src="img/dataframe.jpeg"/>
</div>

# Reglas de descomposición

<div>
<img src="img/descomp.jpeg"/>
</div>

# Operadores
## Map operator
Aplica de forma uniforme un argumento de función a cada partición en paralelo.

🔸 Nota: la función map no debe cambiar la forma de las particiones.

<div>
<img src="img/map.svg"/>
</div>

Este operador tiene mejor rendimiento cuando el número de particiones es igual al número de CPUs, ya que cada partición puede ser procesada en paralelo.

Cuando el número de particiones es aproximadamente 1.5 veces mayor que el número de CPUs, Modin aplica una heurística para combinar particiones, logrando así una repartición “ideal” que permita que cada nueva partición sea procesada en paralelo.

# Operadores
## Reduce operator
Aplica una función argumento que **reduce cada columna o fila** (según el eje especificado) a un escalar, pero requiere conocimiento completo del eje. Notemos que obtener esta información puede ser costoso, ya que el motor de ejecución debe concatenar particiones a lo largo de ese eje.

<div>
<img src="img/reduce.svg"/>
</div>

Este operador tiene mejor rendimiento cuando el número de particiones a lo largo del eje especificado es igual al número de CPUs, permitiendo que cada partición del eje se procese en paralelo.

# Operadores
## TreeReduce operator
Aplica una función que reduce un eje específico a un escalar.
Primero aplica una función map a cada partición en paralelo, luego concatena las particiones resultantes a lo largo del eje y finalmente aplica la función reduce.

🔁 A diferencia del patrón map, acá sí se permite cambiar la forma de las particiones durante la fase map.

⚠️ El motor de ejecución espera que la función reduce retorne un dataframe unidimensional.

💡 El mejor rendimiento se logra cuando el número de particiones (iniciales e intermedias) coincide con la cantidad de CPUs, para que cada partición del eje pueda ser procesada en paralelo.

# Operadores
## Binary operator
Aplica una función que toma exactamente dos operandos, donde el primero siempre es un QueryCompiler.
Si ambos operandos son QueryCompiler, el motor de ejecución transmite (broadcast) las particiones del segundo operando (derecha) al primero (izquierda).

<div>
<img src="img/binary.svg"/>
</div>

⚠️ Advertencia: Para poder hacer broadcasting entre dataframes, la partición a lo largo del eje índice debe coincidir.
Si no coincide, se necesita realinear primero, lo cual implica reparticionar — una operación más costosa que aplicar la función binaria en sí.

💡 Mejor rendimiento:

- Cuando ambos operandos tienen la misma partición

- Y el número de particiones = número de CPUs

# Operadores
## Fold operator
Aplica una función que requiere conocimiento de todo el eje especificado (filas o columnas).

⚠️ Tener acceso a todo el eje puede ser costoso, ya que el motor de ejecución debe concatenar todas las particiones a lo largo de ese eje para ejecutar la operación.

💡 Rendimiento óptimo:

- Cuando el número de particiones (en el eje correspondiente) coincide con la cantidad de CPUs, permitiendo el procesamiento paralelo de cada partición del eje.

## GroupBy operator
Este operador evalúa agregaciones GroupBy que pueden ejecutarse usando el enfoque TreeReduce.

🔄 Para formar los grupos, el motor de ejecución transmite (broadcast) las columnas de agrupamiento (by) a cada partición del DataFrame de origen.

🔍 Etapas:

- Map Stage: calcula la agregación por partición de fila individualmente.

- Reduce Stage: une los resultados, generando un DataFrame con:

    - Filas = número de grupos × número de particiones de fila.

⚠ Si hay demasiados grupos, el DataFrame generado en la etapa de reducción puede ser más grande que el original, afectando el rendimiento.

💡 Rendimiento óptimo:

- Cuando la cardinalidad de las columnas by es baja (pocos grupos resultantes).

## Default-to-pandas operator
Do fallback to pandas for passed function.

This operator has a performance penalty for going from a partitioned Modin DataFrame to pandas because of the communication cost and single-threaded nature of pandas.

# Manejo de los tipos
Al permitir heterogenidadad de tipos a lo largo de las columnas se define la siguiente jerarquia de tipos.

<div>
<img src="img/types.jpeg"/>
</div>

# Manejo de los tipos

**Invariante 1**
Los tipos de columna de salida de los operadores que aceptan una función definida por el usuario (UDF) o función definida por el sistema (SDF) son:

- o bien proporcionados al momento de la invocación,

- o bien designados como UNSPECIFIED y se infieren implícitamente.

La inferencia de tipos se pospone hasta que un operador la requiere.

**Invariante 2**
El tipo $T_i$ de la columna $i$ de un DataFrame es siempre correcto, aunque $T_i$ puede no ser el tipo más preciso para esa columna.

# Manejo de los labels

Este tema presenta un desafío interesante:
el gestor de metadatos debe ser lo suficientemente flexible para permitir que las etiquetas (labels) se **muevan dentro de los datos** (es decir, to_label) y viceversa (es decir, from_label).

Además de la flexibilidad, existen expectativas de baja latencia para las máscaras (mask). Por eso, el sistema debe:

- ejecutar consultas sobre los labels de forma rápida,

- y ser lo bastante flexible para mover los labels dentro de los datos.

Para resolver esto, Modin mantiene **dos conjuntos de labels**:

- Un conjunto cerca de los datos para permitir conversiones rápidas entre etiquetas y datos,

- Otro conjunto externo, como estructura de índice, para soportar consultas basadas en labels.

Modin sincroniza de forma perezosa (lazy) estos dos conjuntos cuando uno cambia y el otro es accedido.

# Manejo del orden logico

Los DataFrames tienen un **orden lógico** que provee una vista consistente de los datos:

- Tras cada transformación, filas y columnas mantienen el mismo orden.

- Cada fila/columna está asociada a un desplazamiento numérico o **posición**.

En sistemas como pandas, el orden lógico y físico están muy ligados.
En cambio, Modin propone un sistema que separa el orden lógico del orden físico:

- Modin mantiene ansiosamente el orden lógico.

- Las posiciones físicas se materializan de forma perezosa (lazy) y solo se calculan cuando se necesitan.

# Instalacion y motor de ejecucion

In [6]:
pip install "modin[all]" -q # Install Modin with Ray and Dask engines.

Note: you may need to restart the kernel to use updated packages.


Si queremos instalar solo un motor en especifico

In [5]:
# pip install "modin[ray]"  --> Install Modin dependencies and Ray.
# pip install "modin[dask]" --> Install Modin dependencies and Dask.
# pip install "modin[mpi]"  --> Install Modin dependencies and MPI through unidist.

Por default el motor es **Ray** para elegir cual queremos podemos hacer:

In [ ]:
import modin.config as modin_cfg
import unidist.config as unidist_cfg

modin_cfg.Engine.put("ray")  # Modin will use Ray
modin_cfg.Engine.put("dask")  # Modin will use Dask

modin_cfg.Engine.put('unidist') # Modin will use Unidist
unidist_cfg.Backend.put('mpi') # Unidist will use MPI backend

> You should not change the engine after your first operation with Modin as it will result in undefined behavior.

# Comparando **read_csv**

In [7]:
import modin.pandas as pd
import pandas
import time
from IPython.display import Markdown, display

def printmd(string):
    display(Markdown(string))

In [8]:
path = 'taxi.csv'

In [9]:
start = time.time()
pandas_df = pandas.read_csv(path, parse_dates=["tpep_pickup_datetime", "tpep_dropoff_datetime"], quoting=3)
end = time.time()
pandas_duration = end - start
print("Time to read with pandas: {} seconds".format(round(pandas_duration, 3)))

Time to read with pandas: 2.923 seconds


In [12]:
import multiprocessing

cores = multiprocessing.cpu_count() # Count the number of cores in a computer
cores

16

In [11]:
start = time.time()
modin_df = pd.read_csv(path, parse_dates=["tpep_pickup_datetime", "tpep_dropoff_datetime"], quoting=3)
end = time.time()
modin_duration = end - start
print("Time to read with Modin: {} seconds".format(round(modin_duration, 3)))

printmd("### Modin is {}x faster than pandas at `read_csv`!".format(round(pandas_duration / modin_duration, 2)))

Time to read with Modin: 1.268 seconds


### Modin is 2.31x faster than pandas at `read_csv`!